# BioTransport CFD Lab

Learning goals: connect random walks, diffusion, advection, reaction, radial geometry, and biosensor response to biological transport examples. The Python solvers are fast teaching models with prescribed flow fields. OpenFOAM is optional for higher-fidelity CFD.

In [ ]:
# Setup cell
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

OUT = ROOT / "outputs" / "notebook"
OUT.mkdir(parents=True, exist_ok=True)

from biotransport_lab.api import simulate_cartesian_preset, simulate_radial_preset
from biotransport_lab.dimensionless import summarize_dimensionless_numbers
from biotransport_lab.random_walk import simulate_random_walk
from biotransport_lab.visualization import (
    save_cartesian_outputs,
    save_radial_outputs,
    save_random_walk_outputs,
    write_json,
)

print(f"Notebook outputs: {OUT}")

## 1. Random walk to diffusion

Molecules move by many small thermal steps. The mean squared displacement links particle motion to the diffusion coefficient.

In [ ]:
# Parameters: random walk
particles = 1200
steps = 200
D = 80.0
dt = 0.01

rw = simulate_random_walk(particles=particles, steps=steps, D_um2_s=D, dt_s=dt, dimensions=2)
save_random_walk_outputs(rw, OUT / "random_walk")
write_json(rw.to_json_summary(), OUT / "random_walk" / "statistics.json")
rw.to_json_summary()["metadata"]

## 2. Fick's law and flux

A localized molecular release spreads down its concentration gradient. With no source or sink, no-flux walls approximately conserve total mass.

In [ ]:
# Parameters: Fickian diffusion
D = 100.0
total_time = 0.6

fick = simulate_cartesian_preset(preset="ficks_law_diffusion", D_um2_s=D, total_time_s=total_time)
save_cartesian_outputs(fick, OUT / "ficks_law_diffusion")
fick.state_label, fick.total_mass[0], fick.total_mass[-1]

## 3. Advection-diffusion in a microchannel

A microfluidic assay can carry target molecules downstream while diffusion spreads the plume across the channel.

In [ ]:
# Parameters: microchannel plume
D = 80.0
U = 200.0
total_time = 0.8

micro = simulate_cartesian_preset(
    preset="microchannel_advection_diffusion",
    D_um2_s=D,
    U_um_s=U,
    k_s=0.0,
    total_time_s=total_time,
)
save_cartesian_outputs(micro, OUT / "microchannel_advection_diffusion")
micro.dimensionless["Pe"]

## 4. Reaction-diffusion

Nutrients, oxygen, ligands, and drugs can be consumed or degraded while diffusing.

In [ ]:
# Parameters: reaction-diffusion sink
D = 80.0
k = 0.03
source_x = 24.0
sensor_x = 170.0

reaction = simulate_cartesian_preset(
    preset="reaction_diffusion_sink",
    D_um2_s=D,
    U_um_s=0.0,
    k_s=k,
    source_x_um=source_x,
    sensor_x_um=sensor_x,
    total_time_s=0.6,
)
save_cartesian_outputs(reaction, OUT / "reaction_diffusion_sink")
reaction.dimensionless["Da_diffusion"]

## 5. Cylindrical radial transport

Cylindrical coordinates are useful for transport around blood vessels, fibers, and long microchannel features. The reported flux is per unit length.

In [ ]:
# Parameters: cylindrical vessel transport
radius = 20.0
D = 120.0

cyl = simulate_radial_preset(
    preset="cylindrical_vessel_transport",
    geometry="cylindrical",
    D_um2_s=D,
    radius_um=radius,
    boundary_kind="noflux",
    total_time_s=0.3,
)
save_radial_outputs(cyl, OUT / "cylindrical_vessel_transport")
cyl.metadata["tau_diffusion_s"], cyl.boundary_flux[-1]

## 6. Spherical radial transport

Spherical coordinates describe cells, spheroids, aggregates, and drug-loaded beads. The spherical boundary flux includes the full surface area.

In [ ]:
# Parameters: spherical cell uptake
radius = 10.0
D = 100.0
outer_concentration = 1.0

sphere = simulate_radial_preset(
    preset="spherical_cell_uptake",
    geometry="spherical",
    D_um2_s=D,
    radius_um=radius,
    outer_concentration=outer_concentration,
    boundary_kind="absorbing",
    total_time_s=0.2,
)
save_radial_outputs(sphere, OUT / "spherical_cell_uptake")
sphere.state_label, sphere.boundary_flux[-1]

## 7. Microchannel biosensor

A sensor patch can absorb or bind target molecules. The sensor response curve is a simplified version of a biosensor breakthrough curve.

In [ ]:
# Parameters: biosensor design
D = 80.0
U = 200.0
k = 0.02
sensor_y = 18.0

biosensor = simulate_cartesian_preset(
    preset="microchannel_biosensor",
    D_um2_s=D,
    U_um_s=U,
    k_s=k,
    sensor_y_um=sensor_y,
    total_time_s=0.8,
)
save_cartesian_outputs(biosensor, OUT / "microchannel_biosensor")
biosensor.sensor_concentration[-1], biosensor.sensor_absorption_flux[-1]

## 8. Dimensionless interpretation

`Pe` compares advection with diffusion, `Da` compares reaction with transport, `Re` estimates microflow inertia, and the Bi-like number compares surface reaction with diffusion.

In [ ]:
# Parameters: dimensionless numbers
numbers = summarize_dimensionless_numbers(
    U_um_s=200.0,
    L_um=100.0,
    D_um2_s=80.0,
    k_s=0.02,
    surface_rate_um_s=1.0,
)
numbers

## 9. Student exercises

Change `D`, `U`, `k`, source position, sensor position, radius, geometry, and boundary condition. Explain what biological change each parameter represents and whether the response is diffusion-limited, advection-limited, or reaction-limited.

## 10. Limitations

The Python solver uses prescribed flow fields, 2D approximations, simple first-order reactions, and classroom-sized grids. Some examples do not reach a true steady state over the chosen simulation time; those outputs are labeled as final simulated state. Use the optional OpenFOAM backend when full CFD detail is needed.